# LTE-LoRA: Training + Evaluation (Final)

**Runtime:** A100 GPU (Runtime > Change runtime type > A100)

Run every cell top-to-bottom. Training takes ~4-8 hours.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi | head -12
!python --version

In [ ]:
# Cell 2: Install dependencies (~2 min)
# We do NOT install flash_attn (takes 15 min to compile and not needed)
# We do NOT pin transformers (use whatever Colab has)
!pip install peft bitsandbytes scipy deepspeed sentence_transformers higher -q
!pip install fschat -q
print('Done!')

In [ ]:
# Cell 3: Clone repo + download data + patch code
import os
os.chdir('/content')
!rm -rf LTE
!git clone https://github.com/YJiangcm/LTE.git
os.chdir('/content/LTE')

# Download training data
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id='YuxinJiang/LTE_train_data', filename='LTE_train_data.json',
                repo_type='dataset', local_dir='data/')
print(f'Training data: {os.path.getsize("data/LTE_train_data.json") / 1e6:.0f} MB')

# ---- PATCH train_lora.py ----
with open('FastChat/fastchat/train/train_lora.py', 'r') as f:
    code = f.read()

# Fix 1: transformers.deepspeed moved in v5
code = code.replace(
    'from transformers import Trainer, BitsAndBytesConfig, deepspeed',
    'from transformers import Trainer, BitsAndBytesConfig\n'
    'try:\n'
    '    from transformers import deepspeed\n'
    'except ImportError:\n'
    '    from transformers.integrations import deepspeed'
)

# Fix 2: flash_attn not installed
code = code.replace(
    'from fastchat.train.llama_flash_attn_monkey_patch import (\n'
    '    replace_llama_attn_with_flash_attn,\n'
    ')',
    'try:\n'
    '    from fastchat.train.llama_flash_attn_monkey_patch import (\n'
    '        replace_llama_attn_with_flash_attn,\n'
    '    )\n'
    'except ImportError:\n'
    '    def replace_llama_attn_with_flash_attn(): pass'
)

# Fix 3: CPUAdam builder may fail
code = code.replace(
    'import deepspeed\ndeepspeed.ops.op_builder.CPUAdamBuilder().load()',
    'try:\n    import deepspeed\n    deepspeed.ops.op_builder.CPUAdamBuilder().load()\nexcept Exception:\n    pass'
)

# Fix 4: torch_dtype deprecation warning
code = code.replace('torch_dtype', 'dtype') if 'torch_dtype' not in 'BitsAndBytesConfig' else code

with open('FastChat/fastchat/train/train_lora.py', 'w') as f:
    f.write(code)

print('Patched!')

In [ ]:
# Cell 4: Pre-download the model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL = 'NousResearch/Llama-2-7b-chat-hf'
tok = AutoTokenizer.from_pretrained(MODEL)
m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16)
print(f'Model: {m.num_parameters()/1e9:.1f}B params')
del m, tok
torch.cuda.empty_cache()

In [ ]:
# Cell 5: TRAIN (4-8 hours)
import os, torch
os.chdir('/content/LTE')
os.environ['WANDB_DISABLED'] = 'true'
torch.cuda.empty_cache()

# batch=2, grad_accum=64 => effective batch=128 (fits A100 40GB)
!python FastChat/fastchat/train/train_lora.py \
    --model_name_or_path NousResearch/Llama-2-7b-chat-hf \
    --lora_r 8 --lora_alpha 16 --lora_dropout 0.05 \
    --data_path data/LTE_train_data.json \
    --bf16 True \
    --output_dir output_lte_lora_llama-2_7b_chat \
    --num_train_epochs 3 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 64 \
    --eval_strategy no \
    --save_strategy steps --save_steps 300 --save_total_limit 2 \
    --learning_rate 3e-4 --weight_decay 0.0 --warmup_ratio 0.03 \
    --lr_scheduler_type cosine --logging_steps 1 \
    --tf32 True --model_max_length 2048 \
    --gradient_checkpointing True \
    --q_lora False --report_to none

In [ ]:
# Cell 6: Verify checkpoint exists
import os
out = '/content/LTE/output_lte_lora_llama-2_7b_chat'
if os.path.exists(os.path.join(out, 'adapter_config.json')):
    print('Training complete! LoRA adapter saved.')
    for f in os.listdir(out):
        full = os.path.join(out, f)
        if os.path.isfile(full):
            print(f'  {f} ({os.path.getsize(full)/1e6:.1f} MB)')
else:
    print('WARNING: No adapter_config.json found!')
    print('Files:', os.listdir(out) if os.path.exists(out) else 'dir missing')

In [ ]:
# Cell 7: Save checkpoint to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/LTE_checkpoints
!cp -r /content/LTE/output_lte_lora_llama-2_7b_chat /content/drive/MyDrive/LTE_checkpoints/
print('Saved to Drive!')

---
## Evaluation (Day 3)

In [ ]:
# Cell 8: Download retrieval models
import os
os.chdir('/content/LTE')
!git clone https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 hugging_cache/all-MiniLM-L6-v2 2>/dev/null
print('Retrieval model ready.')

In [ ]:
# Cell 9: Update hparams for evaluation
import yaml

hp = '/content/LTE/EasyEdit/hparams/IKE/llama-7b.yaml'
with open(hp) as f:
    cfg = yaml.safe_load(f)

cfg['model_name'] = 'NousResearch/Llama-2-7b-chat-hf'
cfg['lora_name'] = '/content/LTE/output_lte_lora_llama-2_7b_chat'
cfg['sentence_model_name'] = '/content/LTE/hugging_cache/all-MiniLM-L6-v2'

with open(hp, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Hparams updated:', cfg)

In [ ]:
# Cell 10: Evaluate on ZsRE
import os
os.chdir('/content/LTE/EasyEdit')
os.makedirs('output', exist_ok=True)

for aspect in ['portability_r', 'portability_s', 'portability_l', 'locality_rs']:
    print(f'\n{"="*50} ZsRE - {aspect} {"="*50}')
    !python run_knowedit.py \
        --editing_method=IKE \
        --hparams_dir=hparams/IKE/llama-7b \
        --data_dir=data/knowedit/ZsRE/ZsRE-test-all.json \
        --eval_aspect=$aspect \
        --fluency

In [ ]:
# Cell 11: Evaluate on WikiDatacounterfact
import os
os.chdir('/content/LTE/EasyEdit')

for aspect in ['portability_r', 'portability_s', 'portability_l', 'locality_rs', 'locality_f']:
    print(f'\n{"="*50} WikiData_cf - {aspect} {"="*50}')
    !python run_knowedit.py \
        --editing_method=IKE \
        --hparams_dir=hparams/IKE/llama-7b \
        --data_dir=data/knowedit/wiki_counterfact/test_cf.json \
        --eval_aspect=$aspect \
        --fluency

In [ ]:
# Cell 12: Collect & display all results
import json, os

output_dir = '/content/LTE/EasyEdit/output'
print('\n' + '='*60)
print('LTE-LoRA RESULTS')
print('='*60)

for fname in sorted(os.listdir(output_dir)):
    if fname.endswith('.json'):
        with open(os.path.join(output_dir, fname)) as f:
            metrics = json.load(f)
        print(f'\n{fname}: {len(metrics)} examples')
        
        # Edit success
        try:
            es = sum(m['post']['rewrite_acc'] for m in metrics) / len(metrics) * 100
            print(f'  Edit Success: {es:.2f}%')
        except: pass
        
        # Portability
        try:
            port = metrics[0]['post'].get('portability', {})
            if port:
                for k in port:
                    avg = sum(m['post']['portability'][k] for m in metrics) / len(metrics) * 100
                    print(f'  Portability ({k}): {avg:.2f}%')
        except: pass
        
        # Locality
        try:
            loc = metrics[0]['post'].get('locality', {})
            if loc:
                for k in loc:
                    avg = sum(m['post']['locality'][k] for m in metrics) / len(metrics) * 100
                    print(f'  Locality ({k}): {avg:.2f}%')
        except: pass
        
        # Fluency
        try:
            if 'fluency' in metrics[0]['post']:
                fl = sum(m['post']['fluency']['ngram_entropy'] for m in metrics) / len(metrics) * 100
                print(f'  Fluency: {fl:.2f}')
        except: pass

# Save to Drive
!cp -r /content/LTE/EasyEdit/output/* /content/drive/MyDrive/LTE_checkpoints/results/ 2>/dev/null
print('\nResults saved to Drive!')